# Experiment design for Indian Hinglish turn detection

This notebook operationalizes the formal [experiment plan](../docs/02_experiment_plan.md). It pre-registers questions, hypotheses, controls, metrics, and decision rules, then checks them against the repository's runnable experiment registry.

**Objective:** identify which data and model choices reduce premature interruption on Hinglish-like pauses and fillers without hiding the trade-off in completion recall, latency, or model size.

**Success for this notebook:** every experiment has a comparator, one causal question, measurable guardrails, reproducible configuration, and structured output contract. This notebook validates design; it does not launch GPU training.

## Reproducibility setup

Design source of truth is `scripts/run_experiments.py`; model defaults come from `configs/baseline.yaml`. Reading these objects directly prevents notebook tables from drifting away from executable experiments.

In [1]:
from pathlib import Path
import json
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")

import polars as pl
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").is_dir():
    raise RuntimeError("Run from repository root or notebooks/ directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from configs import config as cfg
from scripts.run_experiments import (
    CORE_EXPERIMENT_IDS,
    EXPERIMENTS,
    EXPERIMENT_PROTOCOL_VERSION,
    TRACKED_METRICS,
    _apply_overrides,
    build_experiment_manifest,
)

SEED = 42
BASE_CONFIG_PATH = PROJECT_ROOT / "configs" / "baseline.yaml"
BASE_CONFIG = yaml.safe_load(BASE_CONFIG_PATH.read_text(encoding="utf-8"))
pl.Config.set_tbl_rows(20)

print(f"project={PROJECT_ROOT.name}; protocol={EXPERIMENT_PROTOCOL_VERSION}; seed={SEED}")
print(f"registered_experiments={len(EXPERIMENTS)}; core_suite={len(CORE_EXPERIMENT_IDS)}")

project=hinglish-turn-detection; protocol=2; seed=42
registered_experiments=12; core_suite=8


## 1. Research questions

1. **Data value:** does targeted augmentation improve pause/filler safety more than changing architecture?
2. **Pause policy:** which silence range generalizes across brief breaths and long hesitation?
3. **Filler policy:** does label-preserving filler injection add value beyond generic acoustic transforms?
4. **Temporal aggregation:** do attention, mean, and last-frame pooling use endpoint evidence differently?
5. **Adaptation depth:** is full Whisper fine-tuning necessary, or can frozen/partial tuning preserve quality with lower cost?
6. **Semantic value:** do transcript features improve Hinglish hard cases enough to justify ASR latency and alignment constraints?
7. **External validity:** do aggregate results survive Hindi, filler, pause, synthetic/recorded, and manually curated Hinglish slices?

These are causal comparison questions, not a leaderboard sweep. Each intervention needs a direct comparator and an interpretation rule written before results are inspected.

## 2. Registered hypotheses and controlled experiments

E1 is the unaugmented reference. E2 is the full targeted-augmentation control used by most architecture and component ablations. E3-E12 change one decision relative to their named comparator. M1 is kept separate because cached transcripts require unaugmented audio to remain aligned.

In [2]:
registry = pl.DataFrame(
    [
        {
            "experiment": spec.experiment_id,
            "question": spec.research_question,
            "comparator": spec.comparator_id or "reference",
            "intervention": spec.description,
            "hypothesis": spec.hypothesis,
        }
        for spec in EXPERIMENTS
    ]
)
display(registry)

experiment,question,comparator,intervention,hypothesis
str,str,str,str,str
"""E1_no_augmentation""","""data_vs_architecture""","""reference""","""Attention pooling on original,…","""Raw audio establishes how much…"
"""E2_augmented""","""data_vs_architecture""","""E1_no_augmentation""","""Attention pooling with full Hi…","""Targeted augmentation lowers f…"
"""E3_mean_pool""","""pooling""","""E2_augmented""","""Mean pooling with same augment…","""Mean pooling dilutes local end…"
"""E4_last_pool""","""pooling""","""E2_augmented""","""Last-frame pooling with same a…","""Last-frame pooling is brittle …"
"""E5_short_pauses""","""silence_length""","""E2_augmented""","""Silence augmentation restricte…","""Short-pause-only training fail…"
"""E6_long_pauses""","""silence_length""","""E2_augmented""","""Silence augmentation restricte…","""Long-pause-only training over-…"
"""E7_frozen_encoder""","""architecture""","""E2_augmented""","""Freeze all four Whisper encode…","""A frozen ASR encoder lacks tas…"
"""E8_partial_finetune""","""architecture""","""E2_augmented""","""Freeze first two Whisper encod…","""Partial tuning retains most fu…"
"""E9_no_filler""","""filler_injection""","""E2_augmented""","""Full augmentation except fille…","""Filler injection improves fill…"


### Major comparison blocks

| Block | Direct comparisons | Inference allowed |
|---|---|---|
| Data | E2 vs E1 | Total effect of enabled targeted data policy |
| Pooling | E3/E4 vs E2 | Pooling effect under identical augmented training |
| Silence | E5/E6/E11 vs E2 | Range and incremental value of inserted pauses |
| Fillers | E9 vs E2; E10 vs E1 | Incremental and standalone filler value |
| Encoder | E7/E8 vs E2 | Representation adaptation versus trainable capacity |
| Hard mining | E12 vs E2 | Sampling curriculum effect |
| Multimodal | M1 vs E1 | Semantic feature value at matched unaugmented conditions |

Do not compare arbitrary rows and assign causality. For example, E7 vs E3 changes both encoder tuning and pooling, so that difference is descriptive only.

## 3. Control audit

Resolved configurations make hidden differences visible. A direct pair should differ only in fields needed for its intended intervention. E10 is the exception in raw configuration count: it enables augmentation while zeroing every transform except fillers; causally, the active change versus E1 is still filler injection.

In [3]:
def flatten(mapping, prefix=""):
    flat = {}
    for key, value in mapping.items():
        name = f"{prefix}.{key}" if prefix else key
        if isinstance(value, dict):
            flat.update(flatten(value, name))
        else:
            flat[name] = value
    return flat


resolved = {
    spec.experiment_id: _apply_overrides(BASE_CONFIG, spec.overrides)
    for spec in EXPERIMENTS
}
control_rows = []
for spec in EXPERIMENTS:
    if spec.comparator_id is None:
        control_rows.append(
            {"experiment": spec.experiment_id, "comparator": "reference", "changed_fields": "-", "field_count": 0}
        )
        continue
    candidate = flatten(resolved[spec.experiment_id])
    comparator = flatten(resolved[spec.comparator_id])
    changed = sorted(key for key in candidate.keys() | comparator.keys() if candidate.get(key) != comparator.get(key))
    control_rows.append(
        {
            "experiment": spec.experiment_id,
            "comparator": spec.comparator_id,
            "changed_fields": ", ".join(changed),
            "field_count": len(changed),
        }
    )

control_audit = pl.DataFrame(control_rows)
display(control_audit)

experiment,comparator,changed_fields,field_count
str,str,str,i64
"""E1_no_augmentation""","""reference""","""-""",0
"""E2_augmented""","""E1_no_augmentation""","""data.use_augmentation""",1
"""E3_mean_pool""","""E2_augmented""","""model.pooling""",1
"""E4_last_pool""","""E2_augmented""","""model.pooling""",1
"""E5_short_pauses""","""E2_augmented""","""data.augment_config.silence_ms…",1
"""E6_long_pauses""","""E2_augmented""","""data.augment_config.silence_ms…",1
"""E7_frozen_encoder""","""E2_augmented""","""model.freeze_encoder_layers""",1
"""E8_partial_finetune""","""E2_augmented""","""model.freeze_encoder_layers""",1
"""E9_no_filler""","""E2_augmented""","""data.augment_config.p_filler""",1


## 4. Metrics and failure costs

- **Accuracy:** useful sanity check; insufficient under asymmetric error cost.
- **F1:** balances precision and recall for overall checkpoint selection.
- **False Complete Rate (FCR):** `FP / (FP + TN)` when incomplete is negative. This is primary safety metric because false complete means interrupting a user mid-thought.
- **Completion recall:** guardrail against trivial zero-FCR model that predicts every clip incomplete.
- **Latency:** warmed batch-one CPU/GPU inference; multimodal latency must include ASR, not classifier only.
- **Model size:** parameter count and FP32 megabytes; trainable parameters reported separately for freezing experiments.

Precision and ROC-AUC remain diagnostic metrics. Slice metrics are reported only when sample support and both labels make them interpretable.

In [4]:
metric_contract = pl.DataFrame(
    {
        "metric": ["accuracy", "f1", "false_complete_rate", "recall", "latency", "model_size"],
        "role": ["sanity", "selection", "primary safety", "safety guardrail", "deployment", "deployment"],
        "direction": ["higher", "higher", "lower", "higher", "lower", "lower"],
        "reported_on": [
            "overall + slices",
            "overall + slices",
            "overall + pause/filler/hard slices",
            "overall + complete class",
            "CPU/GPU batch 1; end-to-end for M1",
            "parameters + FP32 MB",
        ],
    }
)
display(metric_contract)

required_metrics = {"accuracy", "f1", "false_complete_rate", "latency", "model_size"}
assert required_metrics <= set(TRACKED_METRICS)

metric,role,direction,reported_on
str,str,str,str
"""accuracy""","""sanity""","""higher""","""overall + slices"""
"""f1""","""selection""","""higher""","""overall + slices"""
"""false_complete_rate""","""primary safety""","""lower""","""overall + pause/filler/hard sl…"
"""recall""","""safety guardrail""","""higher""","""overall + complete class"""
"""latency""","""deployment""","""lower""","""CPU/GPU batch 1; end-to-end fo…"
"""model_size""","""deployment""","""lower""","""parameters + FP32 MB"""


## 5. Pre-registered success criteria

Criteria use both improvement targets and harm guardrails. Thresholds are practical effect sizes, not claims of statistical significance. Close results require repeated seeds and paired uncertainty analysis.

In [5]:
success_table = pl.DataFrame(
    [
        {
            "experiment": spec.experiment_id,
            "comparator": spec.comparator_id or "reference",
            "success_rule": spec.success_criteria,
        }
        for spec in EXPERIMENTS
    ]
)
display(success_table)

experiment,comparator,success_rule
str,str,str
"""E1_no_augmentation""","""reference""","""Reference control; no standalo…"
"""E2_augmented""","""E1_no_augmentation""","""FCR improves on hard slices, r…"
"""E3_mean_pool""","""E2_augmented""","""Attention wins by at least 1 F…"
"""E4_last_pool""","""E2_augmented""","""Attention lowers internal/trai…"
"""E5_short_pauses""","""E2_augmented""","""Standard 100-800 ms range lowe…"
"""E6_long_pauses""","""E2_augmented""","""Standard range improves comple…"
"""E7_frozen_encoder""","""E2_augmented""","""Full tuning improves overall a…"
"""E8_partial_finetune""","""E2_augmented""","""Within 1 F1 point and 2 FCR po…"
"""E9_no_filler""","""E2_augmented""","""Full augmentation improves Hin…"


### Decision rules

1. **Adopt** only when primary improvement clears its criterion and guardrails hold.
2. **Reject** when overall gain comes from unacceptable completion-recall loss or deployment cost.
3. **Inconclusive** when direction changes across seeds, slice support is weak, or paired 95% interval includes zero.
4. **Prefer simpler model** when quality is practically tied; latency and model size break ties.
5. **Do not tune on final test.** Select model and threshold on validation, then evaluate frozen finalist once.

Multiple ablations increase false-discovery risk. Core suite is exploratory; only direct finalists repeated across seeds 42-44 support a confirmatory claim.

## 6. Registry integrity checks

These checks fail early if an experiment loses its hypothesis, comparator, success rule, or required metric. They protect the scientific contract before GPU time is spent.

In [6]:
ids = {spec.experiment_id for spec in EXPERIMENTS}
audit = pl.DataFrame(
    {
        "check": [
            "unique experiment IDs",
            "all comparators registered",
            "all hypotheses non-empty",
            "all success criteria non-empty",
            "required metrics registered",
            "core suite is registered",
        ],
        "passed": [
            len(ids) == len(EXPERIMENTS),
            all(spec.comparator_id is None or spec.comparator_id in ids for spec in EXPERIMENTS),
            all(bool(spec.hypothesis.strip()) for spec in EXPERIMENTS),
            all(bool(spec.success_criteria.strip()) for spec in EXPERIMENTS),
            required_metrics <= set(TRACKED_METRICS),
            CORE_EXPERIMENT_IDS <= ids,
        ],
    }
)
display(audit)
assert audit["passed"].all()

check,passed
str,bool
"""unique experiment IDs""",true
"""all comparators registered""",true
"""all hypotheses non-empty""",true
"""all success criteria non-empty""",true
"""required metrics registered""",true
"""core suite is registered""",true


## 7. Multimodal feasibility gate

M1 is feasible only after frozen-ASR transcripts exist for every compared split. Dynamic audio/filler augmentation stays off because cached text would no longer describe transformed audio. Comparator is E1, not E2, so semantic value is not confounded with augmentation.

In [7]:
split_paths = {
    "train": cfg.SUBSET_DIR / "train_split.parquet",
    "validation": cfg.SUBSET_DIR / "val_split.parquet",
    "test": cfg.SUBSET_DIR / "test_split.parquet",
}
feasibility_rows = []
for split, path in split_paths.items():
    if not path.is_file():
        feasibility_rows.append({"split": split, "manifest_exists": False, "transcript_column": False, "non_null_transcripts": 0})
        continue
    schema = pl.read_parquet_schema(path)
    non_null = 0
    if "transcript" in schema:
        non_null = pl.scan_parquet(path).select(pl.col("transcript").is_not_null().sum()).collect().item()
    feasibility_rows.append(
        {"split": split, "manifest_exists": True, "transcript_column": "transcript" in schema, "non_null_transcripts": non_null}
    )

multimodal_gate = pl.DataFrame(feasibility_rows)
display(multimodal_gate)
if not multimodal_gate["transcript_column"].all():
    print("M1 gate closed: run scripts/transcribe_dataset.py before multimodal training.")
else:
    print("M1 transcript gate open; verify coverage equals row count before training.")

split,manifest_exists,transcript_column,non_null_transcripts
str,bool,bool,i64
"""train""",true,false,0
"""validation""",true,false,0
"""test""",true,false,0


M1 gate closed: run scripts/transcribe_dataset.py before multimodal training.


## 8. Run order and leakage control

1. Dry-run manifest: resolve configs and fingerprint all datasets.
2. Run core suite on validation with seed 42; treat findings as exploratory.
3. Add full pooling/fine-tuning/component matrix only where question remains decision-relevant.
4. Repeat finalists and direct controls with seeds 43 and 44.
5. Require same effect direction in at least two of three seeds; report mean and standard deviation.
6. Freeze architecture, augmentation, checkpoint rule, and decision threshold.
7. Run final test once. Evaluate independently curated Hinglish challenge set afterward.

The original architecture ablations selected checkpoints by validation F1 and kept the threshold at 0.5. The later safety study calibrated each seed on validation data under explicit FCR and recall constraints.

## 9. Manifest and logging contract

Every run records resolved config, hashes, data fingerprints, seed, history, selected checkpoint metrics, slice errors, latency, parameter count, and status. JSON is the authoritative nested record; CSV supports comparison; generated Markdown supports review.

In [8]:
missing_manifests = [str(path) for path in split_paths.values() if not path.is_file()]
if missing_manifests:
    print("Manifest dry-run skipped. Prepare data first:", missing_manifests)
    manifest_summary = None
else:
    manifest_summary = build_experiment_manifest(
        BASE_CONFIG,
        CORE_EXPERIMENT_IDS,
        epochs=3,
        train_meta=split_paths["train"],
        val_meta=split_paths["validation"],
        test_meta=split_paths["test"],
    )
    dataset_audit = pl.DataFrame(
        [
            {"split": name, "rows": record["rows"], "sha256_prefix": record["sha256"][:12]}
            for name, record in manifest_summary["datasets"].items()
        ]
    )
    run_audit = pl.DataFrame(
        [
            {
                "experiment": run["experiment_id"],
                "comparator": run["comparator_id"] or "reference",
                "config_sha256_prefix": run["config_sha256"][:12],
            }
            for run in manifest_summary["experiments"]
        ]
    )
    display(dataset_audit)
    display(run_audit)

split,rows,sha256_prefix
str,i64,str
"""train""",6613,"""7a5b7bafe038"""
"""validation""",904,"""d749deebb348"""
"""test""",4890,"""645b63db32cb"""


experiment,comparator,config_sha256_prefix
str,str,str
"""E1_no_augmentation""","""reference""","""6dc3a19302b0"""
"""E2_augmented""","""E1_no_augmentation""","""cb4ffdc82426"""
"""E3_mean_pool""","""E2_augmented""","""0c00876300b1"""
"""E5_short_pauses""","""E2_augmented""","""d8839ed7b903"""
"""E6_long_pauses""","""E2_augmented""","""d2be97057624"""
"""E9_no_filler""","""E2_augmented""","""ceb52f6de01c"""
"""E11_no_silence""","""E2_augmented""","""26d7194b6432"""
"""E12_no_hard_mining""","""E2_augmented""","""e94524b53989"""


In [9]:
artifact_contract = pl.DataFrame(
    {
        "artifact": [
            "experiment_manifest.json",
            "<experiment>/config.yaml",
            "<experiment>/history.json",
            "<experiment>/validation_metrics.json",
            "<experiment>/test_metrics.json",
            "<experiment>/result.json",
            "all_results.json",
            "comparison.csv",
            "docs/generated/<run>_ablation_report.md",
        ],
        "purpose": [
            "pre-registration, config hashes, data fingerprints",
            "resolved runnable configuration",
            "per-epoch optimization trace",
            "default model-selection comparison and slices",
            "one-time frozen final evaluation",
            "compact metrics, latency, size, errors, and provenance",
            "combined machine-readable records",
            "flat paired-comparison table",
            "human-readable table and per-experiment interpretation",
        ],
    }
)
display(artifact_contract)

artifact,purpose
str,str
"""experiment_manifest.json""","""pre-registration, config hashe…"
"""<experiment>/config.yaml""","""resolved runnable configuratio…"
"""<experiment>/history.json""","""per-epoch optimization trace"""
"""<experiment>/validation_metric…","""default model-selection compar…"
"""<experiment>/test_metrics.json""","""one-time frozen final evaluati…"
"""<experiment>/result.json""","""compact metrics, latency, size…"
"""all_results.json""","""combined machine-readable reco…"
"""comparison.csv""","""flat paired-comparison table"""
"""docs/generated/<run>_ablation_…","""human-readable table and per-e…"


## 10. Comparison procedure

Compare only registered pairs. Report absolute metrics and paired deltas in percentage points. For latency and size, report ratios as well as raw values. Include slice support counts so a dramatic result on a tiny slice is not mistaken for robust evidence.

Existing artifacts may be inspected below to verify schema, but they do not alter hypotheses or success criteria in this notebook.

In [10]:
comparison_candidates = [
    PROJECT_ROOT / "experiments" / "protocol_v2_seed42" / "comparison.csv",
    PROJECT_ROOT / "experiments" / "comparison.csv",
]
comparison_path = next((path for path in comparison_candidates if path.is_file()), None)
if comparison_path is None:
    print("No comparison CSV yet. Run scripts/run_experiments.py after dry-run review.")
else:
    comparison = pl.read_csv(comparison_path)
    required_columns = {
        "experiment_id",
        "accuracy",
        "f1",
        "false_complete_rate",
        "model_size_mb_fp32",
    }
    print(
        f"comparison_artifact={comparison_path.relative_to(PROJECT_ROOT)}; "
        f"rows={comparison.height}; required_columns_present={required_columns <= set(comparison.columns)}"
    )
    display(comparison.select("experiment_id").head(20))

comparison_artifact=experiments\protocol_v2_seed42\comparison.csv; rows=9; required_columns_present=True


experiment_id
str
"""E1_no_augmentation"""
"""E2_augmented"""
"""E3_mean_pool"""
"""E4_last_pool"""
"""E5_short_pauses"""
"""E6_long_pauses"""
"""E7_frozen_encoder"""
"""E8_partial_finetune"""
"""E11_no_silence"""


## 11. Interpretation template

For each registered pair, final report should use this chain:

> **Problem → hypothesis → intervention → controlled result → guardrail check → slice evidence → deployment cost → decision.**

Required conclusion labels:

- **Supported:** criterion cleared and guardrails held across confirmatory seeds.
- **Not supported:** expected effect absent or reversed.
- **Inconclusive:** uncertainty, weak slice support, or inconsistent seeds prevents decision.

Negative results remain useful. They rule out complexity, reveal shortcuts, or identify data gaps. Never rewrite hypothesis after seeing final-test results.

## Study status and next action

The core suite and three-seed safety repeat are complete. E4 seed 44 was selected from validation summaries and evaluated once on held-out test data. The next useful step is a fixed, speaker-disjoint human Hinglish challenge set; the existing test set should remain closed to further tuning.